In [9]:
pip install duckdb


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   -- ------------------------------------- 0.8/13.1 MB 8.3 MB/s eta 0:00:02
   --- ------------------------------------ 1.0/13.1 MB 3.1 MB/s eta 0:00:04
   ------- -------------------------------- 2.4/13.1 MB 4.5 MB/s eta 0:00:03
   ---------- ----------------------------- 3.4/13.1 MB 4.3 MB/s eta 0:00:03
   ------------ --------------------------- 4.2/13.1 MB 4.1 MB/s eta 0:00:03
   --------------- ------------------------ 5.0/13.1 MB 4.1 MB/s eta 0:00:02
   ----------------- ---------------------- 5.8/13.1 MB 4.1 MB/s eta 0:00:02
   -------------------- ------------------- 6.8/13.1 MB 4.1 MB/s eta 0:00:02
   ----------------------- ---------------- 7.6/13.1 MB 4.0 MB/s eta 0:00:02
   ------------------------ --------------- 8.1/13.1 MB 4.0 MB/s eta 0:00:02
   --------------------------- ------------ 9.2/13.1 MB 4.0 MB/s eta 0:00:01
   ------------------------------- -------- 10.2/13.1 MB 4.0 MB/s eta 0:00:01
   -

In [1]:
pip install duckdb pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import duckdb

con = duckdb.connect("food_analyzer.duckdb")

In [2]:
con.execute("""
CREATE TABLE food_products AS
SELECT *
FROM read_csv_auto(
    'D:/food datset/en.openfoodfacts.org.products.csv',
    delim='\t'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
con.execute("SELECT COUNT(*) FROM food_products").fetchall()

[(4535553,)]

In [4]:
con.execute("""
COPY food_products
TO 'food_products.parquet'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
import duckdb

con = duckdb.connect()

result = con.execute("""
SELECT
    product_name,
    nutriscore_grade,
    sugars_100g,
    salt_100g
FROM 'food_products.parquet'
LIMIT 10
""").fetchdf()

print(result)

                           product_name nutriscore_grade  sugars_100g  \
0         Limonade artisanale a la rose          unknown          NaN   
1                         M&amp;M white          unknown          NaN   
2                          Chocolate n3          unknown          NaN   
3                        Pâte de fruits          unknown          NaN   
4  Paleta gran reserva - Sierra nevada-          unknown          NaN   
5                                  None             None          NaN   
6                                  None             None          NaN   
7                                  None             None          NaN   
8      Confiture extra citron de Menton          unknown          NaN   
9                                  None             None          NaN   

   salt_100g  
0        NaN  
1        NaN  
2        NaN  
3        NaN  
4        NaN  
5        NaN  
6        NaN  
7        NaN  
8        NaN  
9        NaN  


In [6]:
result = con.execute("""
SELECT
    product_name,
    brands,
    nutriscore_grade,
    sugars_100g,
    salt_100g
FROM 'food_products.parquet'
WHERE product_name ILIKE '%maggi%'
LIMIT 20
""").fetchdf()

print(result)

                                         product_name             brands  \
0                                Protine al formaggio  Micro ingredients   
1                           Maggi Spaghetti Bolognese              Maggi   
2                           Rosmarin Kartoffeln Maggi       Saint Michel   
3                           Maggi Nudeln in Rahmsauce              Maggi   
4                                     Formaggio misto               None   
5                     Wafer al prosciutto e formaggio               None   
6                      Palline al sapore di formaggio               None   
7          Crepes al sapore di prosciutto e formaggio               None   
8                                     Formaggio grana               None   
9                                Formaggio casera dop               None   
10                         Risotto quattro  formaggio         Preferisco   
11                                              Maggi             Nestle   
12          

In [7]:
con.execute("""
CREATE VIEW nutrition_view AS
SELECT
    code,
    product_name,
    brands,
    ingredients_text,
    nutriscore_grade,
    nutriscore_score,
    nova_group,
    "energy-kcal_100g" AS energy_kcal,
    fat_100g,
    sugars_100g,
    proteins_100g,
    salt_100g
FROM 'food_products.parquet'
""")

In [8]:
result = con.execute("""
SELECT
    product_name,
    nutriscore_grade,
    sugars_100g,
    salt_100g,
    (
        100
        - COALESCE(sugars_100g,0)*0.5
        - COALESCE(salt_100g,0)*10
    ) AS health_score
FROM nutrition_view
LIMIT 20
""").fetchdf()

print(result)

                            product_name nutriscore_grade  sugars_100g  \
0          Limonade artisanale a la rose          unknown          NaN   
1                          M&amp;M white          unknown          NaN   
2                           Chocolate n3          unknown          NaN   
3                         Pâte de fruits          unknown          NaN   
4   Paleta gran reserva - Sierra nevada-          unknown          NaN   
5                                   None             None          NaN   
6                                   None             None          NaN   
7                                   None             None          NaN   
8       Confiture extra citron de Menton          unknown          NaN   
9                                   None             None          NaN   
10                                  None             None          NaN   
11                                  6666          unknown          NaN   
12              granola Bio le Chocola